# 3SING* Lambda Search — λ ∈ {1, 20}

Обучение **HierarchicalGenerator** (3SING*) при λ=1 и λ=20 для сравнения с SING.

| Модель | λ | Уже есть? |
|--------|---|----------|
| SING   | 1 | ✅ `lambda_search/sing_lambda_1` |
| SING   | 20 | ✅ `lambda_search/sing_lambda_20` |
| 3SING* | 1 | ❌ обучаем здесь |
| 3SING* | 20 | ❌ обучаем здесь |

## Перед запуском
1. **Runtime → Change runtime type → T4 GPU**
2. `combined_train.pt` в Drive: `MyDrive/sing-data/combined_train.pt`
3. Запускай ячейки по порядку.

In [ ]:
# ── 1. Проверка GPU ────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU недоступен — смени runtime на T4")

In [ ]:
# ── 2. Клонируем репозиторий ───────────────────────────────────────────────
import os

REPO_URL = "https://github.com/valerizabby/sing-learned-structure.git"
REPO_DIR = "/content/sing-learned-structure"

if os.path.exists(REPO_DIR):
    print("Репо уже есть — делаем git pull")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

print("Done:", REPO_DIR)

In [ ]:
# ── 3. Монтируем Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 4. Зависимости ─────────────────────────────────────────────────────────
!pip install -q sparsemax

In [ ]:
# ── 5. Пути и устройство — ДО импортов SingLS ─────────────────────────────
import sys

DRIVE_ROOT       = "/content/drive/MyDrive"
DATA_ON_DRIVE    = os.path.join(DRIVE_ROOT, "sing-data/combined_train.pt")
RESULTS_ON_DRIVE = os.path.join(DRIVE_ROOT, "sing-results")

os.environ["SING_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import SingLS.config.config as cfg
cfg.EXP_PATH = RESULTS_ON_DRIVE

print("SING_DEVICE =", os.environ["SING_DEVICE"])
print("EXP_PATH    =", cfg.EXP_PATH)
print("DEVICE      =", cfg.DEVICE)

In [ ]:
# ── 6. Параметры ───────────────────────────────────────────────────────────
LAMBDA_GRID = [1, 20]
NUM_EPOCHS  = 30
FORCE_RERUN = False  # True = перезапустить уже обученные

print(f"λ grid : {LAMBDA_GRID}")
print(f"Epochs : {NUM_EPOCHS}")

In [ ]:
# ── 7. Grid search 3SING* ─────────────────────────────────────────────────
import logging
from SingLS.config.config import DEVICE, AttentionType, hidden_size, lr, output_size, struct_lr
from SingLS.model.model import MusicGenerator
from SingLS.model.hierarchical_generator import HierarchicalGenerator
from SingLS.model.structure_transformer import StructureTransformer, StructureModel
from SingLS.trainer.train import ModelTrainer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

data = torch.load(DATA_ON_DRIVE, weights_only=False)
print(f"Data loaded: {len(data)} tracks  |  DEVICE: {DEVICE}")

results = {}

for ssm_lambda in LAMBDA_GRID:
    save_name  = f"3sing_lambda_{int(ssm_lambda)}"
    save_dir   = os.path.join(cfg.EXP_PATH, f"meta_info/lambda_search/{save_name}")
    final_ckpt = os.path.join(save_dir, "model_final.pt")
    os.makedirs(save_dir, exist_ok=True)

    if not FORCE_RERUN and os.path.exists(final_ckpt):
        print(f"\nSKIP λ={ssm_lambda}: уже обучена → {final_ckpt}")
        results[ssm_lambda] = final_ckpt
        continue

    print(f"\n{'='*60}")
    print(f" 3SING* λ={ssm_lambda}  ({LAMBDA_GRID.index(ssm_lambda)+1}/{len(LAMBDA_GRID)})")
    print(f"{'='*60}")

    torch.manual_seed(2022)

    generator = MusicGenerator(
        hidden_size=hidden_size,
        output_size=output_size,
        attention_type=AttentionType.ORIGINAL,
    )
    structure_transformer = StructureTransformer(
        d_model=hidden_size, nhead=4, num_layers=2
    )
    structure_model = StructureModel(
        transformer=structure_transformer,
        proj=torch.nn.Linear(hidden_size, hidden_size),
    )
    model = HierarchicalGenerator(
        generator=generator,
        structure_model=structure_model,
    ).to(DEVICE)

    optimizer = torch.optim.Adam([
        {"params": model.generator.parameters(),       "lr": lr},
        {"params": model.structure_model.parameters(), "lr": struct_lr},
    ])

    trainer = ModelTrainer(
        generator=model, optimizer=optimizer, data=data,
        hidden_size=hidden_size, ssm_lambda=ssm_lambda,
    )
    trainer.train_epochs(
        num_epochs=NUM_EPOCHS, full_training=True,
        variable_size_batches=True, save_name=save_name,
    )

    torch.save(model, final_ckpt)
    print(f"Saved → {final_ckpt}")
    results[ssm_lambda] = final_ckpt

print("\n" + "="*60)
print("ГОТОВО")
for lam, path in results.items():
    print(f"  λ={lam:<4} → {path}")

In [ ]:
# ── 8. Кривые обучения ────────────────────────────────────────────────────
import glob
from inference.plot_training import plot_training_curves, print_summary
from IPython.display import Image

csv_files, labels = [], []
for lam in LAMBDA_GRID:
    pattern = os.path.join(cfg.EXP_PATH, f"logs/3sing_lambda_{int(lam)}_*_metrics.csv")
    found = sorted(glob.glob(pattern))
    if found:
        csv_files.append(found[-1])
        labels.append(f"3SING* λ={lam}")
        print_summary(found[-1])

if csv_files:
    out_png = os.path.join(cfg.EXP_PATH, "3sing_lambda_curves.png")
    plot_training_curves(csv_files, labels=labels, out_path=out_png)
    display(Image(out_png))
else:
    print("CSV-логи не найдены")

## После обучения

Модели сохранены на Drive:
```
sing-results/meta_info/lambda_search/
  3sing_lambda_1/model_final.pt
  3sing_lambda_20/model_final.pt
```

Скачай в `experiments/sing-results/meta_info/lambda_search/` и запусти оценку:
```bash
python3 -m inference.reeval_retrained
```